# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

# Display dataset name and description
print(f"Dataset Name: {metadata.name}\nDescription: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

Entities in the dataset are referenced by their `@id`. This ensures all references are consistent.

Let's print available record sets and their fields.

In [ ]:
# List record sets and their fields by @id

record_sets = dataset.record_sets # list of RecordSet objects

for rs in record_sets:
    print(f"RecordSet @id: {rs.id}")
    print(f"RecordSet name: {rs.name if hasattr(rs, 'name') else ''}")
    print("Fields:")
    for field in rs.fields:
        print(f"  Field @id: {field.id} | name: {getattr(field, 'name', '')}")
    print("Columns:")
    for col in rs.columns:
        print(f"  Column @id: {col.id} | name: {getattr(col, 'name', '')}")
    print("-")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis.

Below, we list available `RecordSet` `@id`s and demonstrate loading all record sets into separate DataFrames.

Use the record set and field `@id`s from the overview above.

In [ ]:
# Build a list of record set @ids
record_set_ids = [rs.id for rs in record_sets]
print("Available RecordSets:")
for rs_id in record_set_ids:
    print(f"  {rs_id}")

# Load all record sets into dataframes
dataframes = {}
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df

# Display columns for the first available record set
if record_set_ids:
    first_record_set_id = record_set_ids[0]
    print("Columns for first RecordSet:", dataframes[first_record_set_id].columns.tolist())
    dataframes[first_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)

Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

Choose a numeric field and a grouping field from the record set overview. All entities are referenced by `@id`.

In [ ]:
# Select a record set for analysis
record_set_id = record_set_ids[0] if record_set_ids else None

# Choose a numeric field and a grouping field from the record set overview.
df = dataframes[record_set_id]

# For demonstration, let's find some numeric columns
numeric_field_id = None
group_field_id = None
if record_set_id:
    rs_obj = next(rs for rs in record_sets if rs.id == record_set_id)
    for col in rs_obj.columns:
        # Try to pick a likely numeric field: GAD-7, PHQ-9, PSQ
        if col.name and ('gad7' in col.name.lower() or 'phq9' in col.name.lower() or 'psq' in col.name.lower()):
            numeric_field_id = col.id
            break
    # Pick a demographic grouping field
    for col in rs_obj.columns:
        if col.name and ('gender' in col.name.lower() or 'level_of_education' in col.name.lower()):
            group_field_id = col.id
            break
    print(f"Chosen numeric_field_id: {numeric_field_id}")
    print(f"Chosen group_field_id: {group_field_id}")
else:
    print("No record sets available.")

if record_set_id and numeric_field_id:
    # Perform filtering, normalization, and grouping
    threshold = 10
    if numeric_field_id in df.columns:
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_id} > {threshold}:")
        print(filtered_df.head())
        # Normalize field
        filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"Normalized {numeric_field_id} for filtered records:")
        print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"].head()])
        # Group by a categorical variable
        if group_field_id and group_field_id in filtered_df.columns:
            grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)
            print(f"Grouped data by {group_field_id}:")
            print(grouped_df.head())
    else:
        print(f"Numeric field {numeric_field_id} not found in columns.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

Below, we'll create a histogram of the selected numeric field and, if grouping field is available, a boxplot by grouping.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Use the filtered DataFrame from EDA
if record_set_id and numeric_field_id and (numeric_field_id in df.columns):
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field_id].dropna(), bins=20, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

    # Boxplot by group field
    if group_field_id and group_field_id in df.columns:
        plt.figure(figsize=(10, 4))
        sns.boxplot(x=df[group_field_id], y=df[numeric_field_id])
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- The FAIR² Mental Health Survey Dataset from Kilifi County, Kenya contains rich demographic and psychological indicator data, accessible via Croissant schema and the `mlcroissant` library.
- Using `@id` referencing, the dataset can be dynamically explored, analyzed, and visualized with reproducibility and clarity.
- Filtering and normalizing responses for psychological assessments (e.g., GAD-7 scores) and grouping by demographic attributes allows for efficient AI-ready data analysis.